## This is a Pytorch implementation of the paper “Try to Poison My Deep Learning Data? Nowhere to Hide Your Trajectory Spectrum!”. In order to quickly reproduce this framework, we provide truncated loss trajectory data from backdoor models trained using CIFAR10+ResNet18 and partial backdoor. There is no need to train the model from scratch. (But we give specific training settings: poison rate is 1% and trigger is the white square in the lower right corner of the image.)

In [ ]:
import numpy as np
trajectory_loss_re = np.load(file="./data_loss_trace/trajectory_loss_re.npy")
trajectory_loss_clean_re = np.load(file="./data_loss_trace/trajectory_loss_clean_re.npy")
trajectory_loss_poison_re = np.load(file="./data_loss_trace/trajectory_loss_poison_re.npy")
print(trajectory_loss_re.shape)
print(trajectory_loss_clean_re.shape)
print(trajectory_loss_poison_re.shape)

In [ ]:
import matplotlib.pyplot as plt
poison_trace = []
clean_trace = []
all_trace = []
for i in range(np.asmatrix(trajectory_loss_poison_re).shape[1]):
    f = []
    for j in range(np.asmatrix(trajectory_loss_poison_re).shape[0]):
        f.append(trajectory_loss_poison_re[j][i])
    poison_trace.append(f)

for i in range(np.asmatrix(trajectory_loss_clean_re).shape[1]):
    f = []
    for j in range(np.asmatrix(trajectory_loss_clean_re).shape[0]):
        f.append(trajectory_loss_clean_re[j][i])
    clean_trace.append(f)

for i in range(np.asmatrix(trajectory_loss_re).shape[1]):
    f = []
    for j in range(np.asmatrix(trajectory_loss_re).shape[0]):
        f.append(trajectory_loss_re[j][i])
    all_trace.append(f)

print(np.asmatrix(poison_trace).shape)
print(np.asmatrix(clean_trace).shape)
print(np.asmatrix(all_trace).shape)

def display(loss_trace):
    label_name = ["benign_loss","poison_loss"]
    plt.figure(figsize=(10, 7), dpi=300)
    for k in range(2):
        avg = np.mean(loss_trace[k], axis=1)
        std = np.std(loss_trace[k], axis=1)
        r1 = list(map(lambda x: x[0]-x[1], zip(avg, std)))
        r2 = list(map(lambda x: x[0]+x[1], zip(avg, std)))
        plt.plot(np.arange(len(avg)), avg, label=label_name[k])
        plt.fill_between(np.arange(len(avg)), r1, r2, alpha=0.2)

loss_trace = [clean_trace, poison_trace]
display(loss_trace)
plt.xlabel('Epoch',fontsize=18)
plt.ylabel('Loss',fontsize=18)
plt.legend(loc='upper right',fontsize=18)
plt.show()

In [ ]:
import torch
trajectory_loss_re = torch.from_numpy(np.asmatrix(trajectory_loss_re)).float()
X_train = trajectory_loss_re[:6000]
X_test = trajectory_loss_re[6000:8000]
s = X_train.shape[1]
d = int(s*2/3)
print(d)
class RNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn_1 = torch.nn.LSTM(input_size=s,hidden_size=64*2,num_layers=3,bidirectional=True,batch_first=True)
        self.out_1 = torch.nn.Linear(in_features=64*2, out_features=d)

        self.rnn_2 = torch.nn.LSTM(input_size=d,hidden_size=64*2,num_layers=3,bidirectional=True,batch_first=True,dropout=0.1)
        self.out_2 = torch.nn.Linear(in_features=64*2, out_features=s)

    def forward(self, x):
        #encoder
        output, (h_n, c_n) = self.rnn_1(x)
        output_in_last_timestep = h_n[-1, :, :]
        encoder = self.out_1(output_in_last_timestep)

        #decoder
        output1, (h_n1, c_n1) = self.rnn_2(encoder.view(-1, 1, d))
        output_in_last_timestep1 = h_n1[-1, :, :]
        decoder = self.out_2(output_in_last_timestep1)
        return encoder, decoder
net = RNN()
optimizer = torch.optim.Adam(net.parameters(), lr=0.0005)
loss_F = torch.nn.MSELoss()
best_loss = 1000
# for epoch in range(10):
#
#     #Training
#     net.train()
#     _, pred = net(X_train.view(-1,1,s))
#     loss = loss_F(pred, X_train.view(-1,s))
#     optimizer.zero_grad()
#     loss.backward()
#     optimizer.step()
#     print('Epoch: ', epoch, '| train loss: %.4f' % loss.cpu().data.numpy())
#
#     #Testing
#     net.eval()
#     _, pred1 = net(X_test.view(-1,1,s))
#     loss_e = loss_F(pred1, X_test.view(-1,s))
#     print(' Eval loss: %.4f' % loss_e.cpu().data.numpy())
#     if loss_e < best_loss:
#         best_loss = loss_e
#         torch.save(net.state_dict(),'./LSTM classifiers/Autoencoder.pth')
#         print(" Saving!!")

In [ ]:
net.load_state_dict(torch.load('./LSTM classifiers/Autoencoder.pth'))
net.eval()

pred_clean, _= net(trajectory_loss_re[:len(trajectory_loss_clean_re)].view(-1, 1, s))
pred_clean = np.array(pred_clean.squeeze(1).detach().numpy())

pred_poison, _= net(trajectory_loss_re[len(trajectory_loss_clean_re):].view(-1, 1, s))
pred_poison = np.array(pred_poison.squeeze(1).detach().numpy())

def norm(dataset):
    mean = np.mean(dataset, axis=0)
    std = np.std(dataset, axis=0)
    normalize = (dataset - mean) / std
    return normalize

pred_clean = norm(np.array(pred_clean))
pred_poison = norm(np.array(pred_poison))

pred_clean_fft = np.abs(np.fft.fft(pred_clean))
pred_poison_fft = np.abs(np.fft.fft(pred_poison))

print(pred_clean_fft.shape)
print(pred_poison_fft.shape)
pred_fft = np.concatenate([pred_clean_fft,pred_poison_fft])
print(pred_fft.shape)

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
labels = np.concatenate([np.ones(pred_clean_fft.shape[0],dtype=int),np.zeros(pred_poison_fft.shape[0],dtype=int)])
tsne = TSNE(n_components=2, random_state=0, n_iter=1500)
all_samples_tsne = tsne.fit_transform(pred_fft)
colors = np.array(['red','cornflowerblue'])
plt.figure(figsize=(20, 15), dpi=200)
plt.scatter(all_samples_tsne[:, 0], all_samples_tsne[:, 1], c=colors[labels], cmap='coolwarm', alpha=0.5)
for spine in plt.gca().spines.values():
    spine.set_linewidth(2)
# plt.xticks([])
# plt.yticks([])
plt.show()

In [20]:
from sklearn.cluster import DBSCAN
import matplotlib.pyplot as plt
cluster_data = all_samples_tsne
eps = 5
min_samples = 5
thecluster = DBSCAN(eps=eps, min_samples=min_samples).fit(cluster_data)
labels = thecluster.labels_

In [ ]:
c1,c2 = 0,0
for i in range(len(pred_poison)):
    if labels[len(pred_clean)+i]==0:
        c1 += 1
for i in range(len(pred_clean)):
    if labels[i]==0:
        c2+=1
print("Acc: {:.3f}% {}/{}".format((len(pred_poison)-c1)/len(pred_poison)*100,len(pred_poison)-c1,len(pred_poison)))
print("FPR: {:.3f}% {}/{}".format((len(pred_clean)-c2)/len(pred_clean)*100,len(pred_clean)-c2,len(pred_clean)))